In [118]:
import pandas as pd 
from pathlib import Path 
Data_path= Path("..")/"data"
raw_path = Data_path/"raw"/"frailty_raw.csv"
df = pd.read_csv(raw_path)

In [119]:
df.head()

,Height,Weight,Age,Grip,Frailty
0,65.8,112,30,30,N
1,71.5,136,19,31,N
2,69.4,153,45,29,N
3,68.2,142,22,28,Y
4,67.8,144,29,24,Y


In [120]:
df.describe()

,Height,Weight,Age,Grip
count,10.000000,10.000000,10.000000,10.000000
mean,68.600000,131.900000,32.500000,26.000000
std,1.670662,14.231811,12.860361,4.521553
min,65.800000,112.000000,17.000000,19.000000
25%,67.825000,120.750000,22.250000,22.500000
50%,68.450000,136.000000,29.500000,27.000000
75%,69.700000,141.750000,43.500000,29.750000
max,71.500000,153.000000,51.000000,31.000000


In [121]:
df.info()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Height   10 non-null     float64
 1   Weight   10 non-null     int64  
 2   Age      10 non-null     int64  
 3   Grip     10 non-null     int64  
 4   Frailty  10 non-null     str    
dtypes: float64(1), int64(3), str(1)
memory usage: 532.0 bytes


(10, 5)

Part A - Unit Standardization

In [122]:
df["Height"] = df["Height"] * 0.0254

df["Weight"] = df["Weight"] *0.453959237

df.head()

,Height,Weight,Age,Grip,Frailty
0,1.67132,50.843435,30,30,N
1,1.81610,61.738456,19,31,N
2,1.76276,69.455763,45,29,N
3,1.73228,64.462212,22,28,Y
4,1.72212,65.370130,29,24,Y


Part B - Feature Engineering

In [123]:
df["BMI"] = (df["Weight"] / (df ["Height"]**2)).round(2)

df.loc[df["Age"]<30, "AgeGroup"] = "<30"
df.loc[(df["Age"]>=30) & (df["Age"]<=45), "AgeGroup"] = "30-45"
df.loc[(df["Age"]>45) & (df["Age"]<=60), "AgeGroup"] = "46-60"
df.loc[df["Age"]>60, "AgeGroup"] = ">60"

df.head()

,Height,Weight,Age,Grip,Frailty,BMI,AgeGroup
0,1.67132,50.843435,30,30,N,18.20,30-45
1,1.81610,61.738456,19,31,N,18.72,<30
2,1.76276,69.455763,45,29,N,22.35,30-45
3,1.73228,64.462212,22,28,Y,21.48,<30
4,1.72212,65.370130,29,24,Y,22.04,<30


Part 3 - Categorical to Numerical Encoding

In [124]:
#https://www.geeksforgeeks.org/pandas/python-pandas-get_dummies-method/ 
# https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html
# https://pandas.pydata.org/docs/user_guide/reshaping.html#reshaping-dummies--> using this for the encoding
# using the repalce fucntion we learned in class
df["Frailty"] = df["Frailty"].replace({"Y":1, "N":0})
df = pd.get_dummies(df,columns=['AgeGroup'],dtype="int8")
df.head()

,Height,Weight,Age,Grip,Frailty,BMI,AgeGroup_30-45,AgeGroup_46-60,AgeGroup_<30
0,1.67132,50.843435,30,30,0,18.20,1,0,0
1,1.81610,61.738456,19,31,0,18.72,0,0,1
2,1.76276,69.455763,45,29,0,22.35,1,0,0
3,1.73228,64.462212,22,28,1,21.48,0,0,1
4,1.72212,65.370130,29,24,1,22.04,0,0,1


Part D - EDA & Reporting

In [125]:
numeric_columns = df.select_dtypes(include="number").columns.tolist() 

summary = df[numeric_columns].describe().T[["mean","50%","std"]]
summary = summary.rename(columns={"50%": "median"}).round(2)
summary

,mean,median,std
Height,1.74,1.74,0.04
Weight,59.88,61.74,6.46
Age,32.50,29.50,12.86
Grip,26.00,27.00,4.52
BMI,19.70,19.20,1.78
AgeGroup_30-45,0.30,0.00,0.48
AgeGroup_46-60,0.20,0.00,0.42
AgeGroup_<30,0.50,0.50,0.53


In [126]:
#https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html
#https://www.w3schools.com/python/pandas/pandas_correlations.asp
grip_corr_frailty=df["Grip"].corr(df["Frailty"])
print("The Correlation between Grip and Frailty was",round(grip_corr_frailty,2))

The Correlation between Grip and Frailty was -0.48


In [127]:
clean_path = Data_path/"raw"/"frailty_clean.csv"
df.round(2).to_csv(clean_path,index=False)